# Vercel AI Gateway: the route, one part at a time

Models other than Claude reach the harness through **Vercel AI Gateway**, which serves one format, OpenAI Chat Completions, and translates it to each model's own API. Claude keeps the Anthropic format and its own transports, the direct client or Claude Code. A **model router** picks which of the two serves each model, for an agent and for each of its sub-agents.

A host plugs it in with one router:

```python
router = ModelRouter(claude=lambda model, retry: AnthropicProvider(retry_policy=retry),  # or ClaudeCodeProvider(...)
                     others=VercelGateway.from_env())                                    # AI_GATEWAY_API_KEY
agent = AnthropicAgent(model=m, provider_value=router.provider_for(m))
```

**Offline first.** Sections 1 to 12 drive the route against a scripted stand-in for both APIs: no network, no key, no model call. Section 13 is an opt-in live run on the gateway, on its free models and then on models the free tier's $5 a month covers.

| Piece | What it does | File |
|---|---|---|
| `ModelRouter` | Which provider serves each model, for an agent and each sub-agent | `agent_base/providers/router.py` |
| `VercelGateway` | The gateway's key and one shared HTTP client; a provider for every model but Claude | `agent_base/providers/vercel_gateway/gateway.py` |
| `OpenAIChatProvider` | The `Provider` protocol over the OpenAI Chat Completions format: requests, stream translation, retries, errors | `agent_base/providers/openai_chat/provider.py` |
| Model catalog | Which format each model speaks, its limits and inputs, data retention, and the live tool check's result | `agent_base/core/model_catalog.py` |
| Billing | A gateway step costs what Vercel charged; the price row is the fallback | `agent_base/pricing/calculator.py` |

The plan behind it is `mental_model/planned_items/multi-provider-models.md`.

In [1]:
from dotenv import load_dotenv
import asyncio
import json
import os

load_dotenv()

import anthropic
import httpx

from agent_base.core.messages import Message
from agent_base.core.provider import RetryPolicy, provider_for_child
from agent_base.core.model_catalog import DEFAULT_CATALOG
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.providers.anthropic.provider import AnthropicProvider
from agent_base.providers.openai_chat import OpenAIChatLLMConfig, OpenAIChatProvider
from agent_base.providers.router import ModelRouter
from agent_base.providers.vercel_gateway import VercelGateway

print("httpx    :", httpx.__version__)
print("anthropic:", anthropic.__version__)
print("gateway key in the environment:", bool(os.environ.get("AI_GATEWAY_API_KEY")), "(only section 13 needs it)")

httpx    : 0.28.1
anthropic: 0.117.0
gateway key in the environment: True (only section 13 needs it)


In [2]:
# ── Test Utils ─────────────────────────────────────────────────────
#
# Wire stands in for both APIs as an httpx mock transport: each request to .../chat/completions or .../messages takes
# the next scripted response for its API, and every request is kept to look at. A streaming request gets the reply as
# server-sent events; a non-streaming one (agent.run with no stream attached) gets it as one JSON body. The router below sends Claude to a direct client
# and every other model to a gateway, both on the scripted transport.

GEMINI, CLAUDE = "google/gemini-2.5-flash", "claude-sonnet-5-5"
FAST = RetryPolicy(max_retries=3, base_delay=0.0)


def chunk(*, content=None, reasoning=None, reasoning_details=None, tool_calls=None, finish=None, usage=None,
          model=GEMINI):
    delta = {k: v for k, v in (("content", content), ("reasoning", reasoning),
                               ("reasoning_details", reasoning_details), ("tool_calls", tool_calls)) if v is not None}
    data = {"id": "gen_1", "object": "chat.completion.chunk", "model": model,
            "choices": [{"index": 0, "delta": delta, "finish_reason": finish}]}
    return {**data, "usage": usage} if usage else data


def stream(*chunks):
    body = "".join(f"data: {json.dumps(c)}\n\n" for c in chunks) + "data: [DONE]\n\n"
    return httpx.Response(200, headers={"content-type": "text/event-stream"}, content=body.encode())


def say(text, *, model=GEMINI, cost=None):
    usage = {"prompt_tokens": 12, "completion_tokens": 3}
    if cost is not None:
        usage |= {"cost": cost, "is_byok": False}
    return stream(chunk(content=text, model=model), chunk(finish="stop", usage=usage, model=model))


def calls(*items, model=GEMINI):
    raw = [{"index": i, "id": cid, "type": "function", "function": {"name": name, "arguments": json.dumps(args)}}
           for i, (cid, name, args) in enumerate(items)]
    return stream(chunk(tool_calls=raw, model=model), chunk(finish="tool_calls", model=model,
                                                            usage={"prompt_tokens": 20, "completion_tokens": 9}))


def chat_error(status, message, kind):
    return httpx.Response(status, json={"error": {"message": message, "type": kind, "code": kind}})


def claude_says(blocks, stop_reason):
    return httpx.Response(200, json={"id": "msg_1", "type": "message", "role": "assistant", "model": CLAUDE,
                                     "content": blocks, "stop_reason": stop_reason, "stop_sequence": None,
                                     "usage": {"input_tokens": 10, "output_tokens": 20}})


def _as_events(response):
    m = response.json()
    ev = [("message_start", {"type": "message_start", "message": {**{k: v for k, v in m.items() if k != "content"},
          "content": [], "stop_reason": None, "usage": {**m["usage"], "output_tokens": 1}}})]
    for i, b in enumerate(m["content"]):
        if b["type"] == "text":
            start, deltas = {"type": "text", "text": ""}, [{"type": "text_delta", "text": b["text"]}]
        else:
            start = {"type": "tool_use", "id": b["id"], "name": b["name"], "input": {}}
            deltas = [{"type": "input_json_delta", "partial_json": json.dumps(b["input"])}]
        ev.append(("content_block_start", {"type": "content_block_start", "index": i, "content_block": start}))
        ev += [("content_block_delta", {"type": "content_block_delta", "index": i, "delta": d}) for d in deltas]
        ev.append(("content_block_stop", {"type": "content_block_stop", "index": i}))
    ev += [("message_delta", {"type": "message_delta", "delta": {"stop_reason": m["stop_reason"]},
                              "usage": {"output_tokens": 20}}), ("message_stop", {"type": "message_stop"})]
    body = "".join(f"event: {n}\ndata: {json.dumps(d)}\n\n" for n, d in ev)
    return httpx.Response(200, headers={"content-type": "text/event-stream"}, content=body.encode())


def _as_json(response):
    message, tool_calls, out, finish = {"role": "assistant", "content": None}, {}, {}, None
    for line in response.content.decode().splitlines():
        if line.startswith("data: {"):
            d = json.loads(line[6:])
            out |= {"id": d["id"], "model": d["model"], **({"usage": d["usage"]} if d.get("usage") else {})}
            for ch in d["choices"]:
                delta = ch["delta"]
                for key in ("content", "reasoning"):
                    if delta.get(key):
                        message[key] = (message.get(key) or "") + delta[key]
                if delta.get("reasoning_details"):
                    message.setdefault("reasoning_details", []).extend(delta["reasoning_details"])
                for c in delta.get("tool_calls") or []:
                    tool_calls[c["index"]] = {k: v for k, v in c.items() if k != "index"}
                finish = ch["finish_reason"] or finish
    if tool_calls:
        message["tool_calls"] = [tool_calls[i] for i in sorted(tool_calls)]
    return httpx.Response(200, json={**out, "object": "chat.completion",
                                     "choices": [{"index": 0, "message": message, "finish_reason": finish}]})


class Wire:
    def __init__(self, chat=(), claude=()):
        self.scripts = {"chat": list(chat), "anthropic": list(claude)}
        self.requests = []
        self.transport = httpx.MockTransport(self._handle)

    def _handle(self, request):
        self.requests.append(request)
        kind = "chat" if request.url.path.endswith("/chat/completions") else "anthropic"
        response = self.scripts[kind].pop(0)
        streaming = json.loads(request.content or b"{}").get("stream", False)
        events = "event-stream" in response.headers.get("content-type", "")
        if response.status_code == 200 and kind == "anthropic" and streaming and not events:
            return _as_events(response)
        if response.status_code == 200 and kind == "chat" and not streaming and events:
            return _as_json(response)
        return response

    def bodies(self, kind="chat"):
        suffix = "/chat/completions" if kind == "chat" else "/messages"
        return [json.loads(r.content) for r in self.requests if r.url.path.endswith(suffix)]

    def gateway(self):
        return VercelGateway(api_key="gateway-key", transport=self.transport)

    def router(self):
        direct = anthropic.AsyncAnthropic(api_key="direct-key", max_retries=0,
                                          http_client=anthropic.DefaultAsyncHttpxClient(transport=self.transport))
        return ModelRouter(claude=lambda model, retry: AnthropicProvider(client=direct, retry_policy=retry or FAST),
                           others=self.gateway())


class Frames:
    """A DeltaSink that prints each frame as it lands."""
    def __init__(self, quiet=False):
        self.deltas, self.quiet = [], quiet

    def emit(self, d):
        self.deltas.append(d)
        if not self.quiet:
            body = getattr(d, "text", None) or getattr(d, "thinking", None) or getattr(d, "arguments_json", "")
            print(f"  {type(d).__name__:14} final={d.is_final!s:5} {body!r}")

    def emit_meta(self, body):
        pass


def show(obj, keys=None):
    if keys:
        obj = {k: obj[k] for k in keys if k in obj}
    print(json.dumps(obj, indent=1, default=str)[:3000])

print("ready")

ready


# 1. Which provider serves a model

The router asks the **catalog** which format a model speaks: a record decides; otherwise `claude-…` is the Anthropic format and any other `creator/model` is Chat Completions. Claude goes to the host's Claude transport (here the direct client); every other model goes to the gateway. A gateway's id for Claude (`anthropic/…`) is refused, with Anthropic's own id in the error, so Claude never reaches Vercel.

Every provider the router hands out carries it (`model_router`), which is how a sub-agent finds its own route (section 10).

In [3]:
router = Wire().router()
for model in (CLAUDE, GEMINI, "openai/gpt-5-nano", "moonshotai/kimi-k3", "anthropic/claude-haiku-5.5", "gpt-5"):
    try:
        p = router.provider_for(model)
        print(f"{model:30} -> {type(p).__name__:20} format={p.model_format}")
    except Exception as e:
        print(f"{model:30} -> refused: {type(e).__name__}: {str(e)[:110]}")

claude-sonnet-5-5              -> AnthropicProvider    format=anthropic
google/gemini-2.5-flash        -> GatewayChatProvider  format=chat_completions
openai/gpt-5-nano              -> GatewayChatProvider  format=chat_completions
moonshotai/kimi-k3             -> GatewayChatProvider  format=chat_completions
anthropic/claude-haiku-5.5     -> refused: UnknownModelError: 'anthropic/claude-haiku-5.5' is a gateway's id for Claude. Claude goes through Anthropic, direct or Claude Cod
gpt-5                          -> refused: UnknownModelError: No catalog record for model 'gpt-5', and its id is neither claude-… nor creator/model. Add a ModelRecord for i


# 2. The catalog

One record per model the library has run through its live tier: its limits and inputs as Vercel lists them, whether Vercel offers zero data retention for it, and **`tools`**, the result of the live tool round (`True` passed, `False` failed). A model without a record still works by its id's form; section 7 shows what its record changes.

In [4]:
print(f"{'model':38} {'context':>9} {'output':>9} {'inputs':16} {'ZDR':5} tools")
for mid in DEFAULT_CATALOG.ids():
    r = DEFAULT_CATALOG.get(mid)
    print(f"{mid:38} {r.context_tokens:>9,} {r.output_tokens:>9,} {','.join(sorted(r.inputs)):16} "
          f"{str(r.zero_data_retention):5} {r.tools}")

model                                    context    output inputs           ZDR   tools
alibaba/qwen3.7-flash                    991,000    64,000 image,pdf,text   True  False
amazon/nova-lite                         300,000     8,192 image,pdf,text   True  True
deepseek/deepseek-v3.1                   163,840   128,000 text             False True
google/gemini-2.5-flash                1,000,000    65,535 image,pdf,text   True  True
meta/muse-ember                        1,048,576 1,048,576 image,pdf,text   True  True
mistral/mistral-nemo                     131,072   131,072 text             False False
nvidia/nemotron-3.5-lightning-free     1,000,000    65,000 text             False True
openai/gpt-5-nano                        400,000   128,000 image,pdf,text   True  True
openai/gpt-oss-20b                       131,072     8,192 text             True  True
recursive/flint-v1.0                     262,144   262,144 text             False True
zai/glm-4.7-flash                       

# 3. One request, built

`build_request` turns the harness's history into a Chat Completions body. The system prompt goes first; `effort` becomes `reasoning.effort`; the tools become functions; and the gateway's own settings ride `providerOptions.gateway`: automatic prompt caching on every request, plus any fallback models or provider order the config names.

In [5]:
from agent_base.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b

wire = Wire()
gemini = wire.gateway().provider_for(GEMINI)
schemas = AnthropicAgent(model=GEMINI, provider_value=gemini, tools=[add], system_prompt="x").tool_registry.get_schemas()
body = gemini.build_request(
    system_prompt="Use the add tool for every sum.", messages=[Message.user("What is 19 + 23?")],
    tool_schemas=schemas, llm_config=OpenAIChatLLMConfig(effort="low", max_tokens=2000,
                                                          fallback_models=["openai/gpt-5-nano"]),
    model=GEMINI, stream=True)
show(body)

{
 "model": "google/gemini-2.5-flash",
 "messages": [
  {
   "role": "system",
   "content": "Use the add tool for every sum."
  },
  {
   "role": "user",
   "content": "What is 19 + 23?"
  }
 ],
 "stream": true,
 "stream_options": {
  "include_usage": true
 },
 "tools": [
  {
   "type": "function",
   "function": {
    "name": "add",
    "description": "Add two integers and return the sum.",
    "parameters": {
     "type": "object",
     "properties": {
      "a": {
       "type": "integer",
       "description": "Parameter of type integer"
      },
      "b": {
       "type": "integer",
       "description": "Parameter of type integer"
      }
     },
     "required": [
      "a",
      "b"
     ]
    }
   }
  }
 ],
 "max_tokens": 2000,
 "reasoning": {
  "effort": "low"
 },
 "providerOptions": {
  "gateway": {
   "caching": "auto",
   "models": [
    "openai/gpt-5-nano"
   ]
  }
 }
}


# 4. A streamed reply

The provider reads the gateway's chunks and puts out the frames the Anthropic provider does: text (and reasoning) as it arrives, then one closing frame per block. The turn ends with the message, the stop reason and usage, and each step keeps the gateway's generation id.

In [6]:
wire = Wire(chat=[stream(chunk(reasoning="The sky scatters blue light."), chunk(content="The sky "),
                         chunk(content="is blue."),
                         chunk(finish="stop", usage={"prompt_tokens": 14, "completion_tokens": 9,
                                                     "completion_tokens_details": {"reasoning_tokens": 6}}))])
turn = await wire.gateway().provider_for(GEMINI).generate_stream(
    system_prompt="Answer in one sentence.", messages=[Message.user("What colour is the sky?")], tool_schemas=[],
    llm_config=None, model=GEMINI, sink=Frames(), agent_uuid="nb")
print("\nstop:", turn.message.stop_reason, "| usage:", turn.message.usage.input_tokens, "in,",
      turn.message.usage.output_tokens, "out,", turn.message.usage.thinking_tokens, "thinking")
print("usage_kwargs:", turn.message.usage_kwargs)

2026-11-18T11:53:49.122923Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


  ThinkingDelta  final=False 'The sky scatters blue light.'
  ThinkingDelta  final=True  ''
  TextDelta      final=False 'The sky '
  TextDelta      final=False 'is blue.'
  TextDelta      final=True  ''

stop: end_turn | usage: 14 in, 9 out, 6 thinking
usage_kwargs: {'finish_reason': 'stop', 'generation_id': 'gen_1', 'gateway': 'vercel_gateway'}


# 5. The whole loop, offline

`AnthropicAgent(provider_value=...)` runs a tool round on a gateway model exactly as on Claude: the model calls `add`, the turn loop runs it, and the result goes back as a `tool` message matched to the call's id.

In [7]:
wire = Wire(chat=[calls(("call_1", "add", {"a": 19, "b": 23})), say("42")])
agent = AnthropicAgent(model=GEMINI, provider_value=wire.router().provider_for(GEMINI), tools=[add],
                       system_prompt="Use the add tool for every sum. Reply with the number only.")
result = await agent.run("What is 19 + 23?")
print("answer:", result.final_answer, "| steps:", result.total_steps)
print("the second request's last two messages:")
show(wire.bodies()[1]["messages"][-2:])

2026-11-18T11:53:49.132451Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T11:53:49.136447Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


answer: 42 | steps: 2
the second request's last two messages:
[
 {
  "role": "assistant",
  "content": null,
  "tool_calls": [
   {
    "id": "call_1",
    "type": "function",
    "function": {
     "name": "add",
     "arguments": "{\"a\": 19, \"b\": 23}"
    }
   }
  ]
 },
 {
  "role": "tool",
  "tool_call_id": "call_1",
  "content": "42"
 }
]


# 6. Reasoning goes back only to the model that wrote it

A model's reasoning comes back as `reasoning_details` (text, summaries, signatures, encrypted data). The provider keeps it on the turn's thinking block and sends it back **only to the same model**: another model's turn goes back as its text and tool calls alone, so no model sees reasoning it can't read.

In [8]:
wire = Wire(chat=[stream(chunk(reasoning_details=[{"type": "reasoning.encrypted", "data": "sig-1", "index": 0}]),
                         chunk(tool_calls=[{"index": 0, "id": "call_1", "type": "function",
                                            "function": {"name": "add", "arguments": '{"a": 1, "b": 2}'}}]),
                         chunk(finish="tool_calls")),
                  say("3"), say("Still 3."), say("3, again.", model="openai/gpt-5-nano")])
router = wire.router()
agent = AnthropicAgent(model=GEMINI, provider_value=router.provider_for(GEMINI), tools=[add], system_prompt="Add.")
await agent.run("1 + 2?")
await agent.run("Sure?")
print("to Gemini, the model that wrote it:", "reasoning_details" in wire.bodies()[2]["messages"][2])

nano = AnthropicAgent(model="openai/gpt-5-nano", provider_value=router.provider_for("openai/gpt-5-nano"),
                      tools=[add], system_prompt="Add.")
await nano.initialize()
nano.agent_config.context_messages = list(agent.agent_config.context_messages)
await nano.run("And you?")
print("to GPT-5 nano, another model:    ", "reasoning_details" in wire.bodies()[3]["messages"][2])

2026-11-18T11:53:49.148299Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T11:53:49.150316Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T11:53:49.153315Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T11:53:49.157313Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


to Gemini, the model that wrote it: True
to GPT-5 nano, another model:     False


# 7. Tools only where the live tool check passed

Some models skip a tool and answer themselves, sometimes wrongly, and some services return a model's call as text: silent failures. So a request with tools goes to a model only as its record allows:

| The record | The provider |
|---|---|
| `tools=True` | sends it |
| `tools=False` | refuses before sending |
| no record | sends it, and logs `openai_chat_tools_unverified` once |

**`tool_choice`** (`"required"`, `"none"` or a tool's name) forces the first step only: once a tool result comes back the model chooses, so an agent can still end its turn.

In [9]:
from agent_base.core.types import ToolResultContent, ToolUseContent

p = OpenAIChatProvider(api_key="k", base_url="https://gateway.example/v1")
for model in (GEMINI, "zai/glm-4.7-flash", "moonshotai/kimi-k3"):
    try:
        b = p.build_request(system_prompt=None, messages=[Message.user("hi")], tool_schemas=schemas,
                            llm_config=None, model=model, stream=False)
        print(f"{model:22} tools sent: {[t['function']['name'] for t in b['tools']]}")
    except Exception as e:
        print(f"{model:22} refused: {str(e)[:120]}")

first = [Message.user("What is 2 + 2?")]
after = first + [Message.assistant([ToolUseContent(tool_name="add", tool_id="c1", tool_input={"a": 2, "b": 2})]),
                 Message.user([ToolResultContent(tool_name="add", tool_id="c1", tool_result="4")])]
forced = OpenAIChatLLMConfig(tool_choice="required")
for label, messages in (("first step", first), ("after a result", after)):
    b = p.build_request(system_prompt=None, messages=messages, tool_schemas=schemas, llm_config=forced,
                        model=GEMINI, stream=False)
    print(f"tool_choice='required', {label:15}: {b.get('tool_choice', '(not sent)')}")

2026-11-18T11:53:49.167321Z [warning  ] openai_chat_tools_unverified   [agent_base.providers.openai_chat.provider] model=moonshotai/kimi-k3 reason='not in the catalog'


google/gemini-2.5-flash tools sent: ['add']
zai/glm-4.7-flash      refused: 'zai/glm-4.7-flash' failed the live tool check (it answers without its tools, or its service returns calls as text), so 
moonshotai/kimi-k3     tools sent: ['add']
tool_choice='required', first step     : required
tool_choice='required', after a result : (not sent)


# 8. Documents follow the catalog too

A PDF goes as a `file` part to a model whose record lists PDF input; a model whose record doesn't is refused, and a model with no record gets it with one warning (`openai_chat_documents_unverified`).

In [10]:
from agent_base.core.types import DocumentContent, TextContent

doc = DocumentContent(media_type="application/pdf", source_type="base64", data="JVBERi0=", filename="report.pdf")
for model in (GEMINI, "deepseek/deepseek-v3.1", "moonshotai/kimi-k3"):
    try:
        b = p.build_request(system_prompt=None, messages=[Message.user([doc, TextContent(text="Summarize.")])],
                            tool_schemas=[], llm_config=None, model=model, stream=False)
        print(f"{model:24} sent: {b['messages'][0]['content'][0]['type']}")
    except Exception as e:
        print(f"{model:24} refused: {str(e)[:110]}")

2026-11-18T11:53:49.174509Z [warning  ] openai_chat_documents_unverified [agent_base.providers.openai_chat.provider] model=moonshotai/kimi-k3 reason='not in the catalog'


google/gemini-2.5-flash  sent: file
deepseek/deepseek-v3.1   refused: 'deepseek/deepseek-v3.1' doesn't take PDF input (its catalog record lists ['text']). Send the document's text 
moonshotai/kimi-k3       sent: file


# 9. Claude never answers in this format

Claude goes through Anthropic only. The gateway refuses a Claude id, and the provider refuses one as the model or as a fallback (`fallback_models`, or `providerOptions.gateway.models` in `api_kwargs`), where the gateway would otherwise fall back to Claude in Chat Completions. A Claude config given to a gateway model is converted: what changes the agent (server tools) is refused, what only tunes an Anthropic request is dropped.

In [11]:
tries = {
    "gateway.provider_for('anthropic/claude-haiku-5.5')": lambda: Wire().gateway().provider_for("anthropic/claude-haiku-5.5"),
    "fallback_models=['claude-haiku-5-5']": lambda: p.build_request(
        system_prompt=None, messages=[Message.user("hi")], tool_schemas=[], model=GEMINI, stream=False,
        llm_config=OpenAIChatLLMConfig(fallback_models=["claude-haiku-5-5"])),
    "AnthropicLLMConfig(server_tools=[web_search])": lambda: p.make_llm_config(
        AnthropicLLMConfig(server_tools=[{"type": "web_search_20260209", "name": "web_search"}])),
}
for label, attempt in tries.items():
    try:
        attempt()
        print(f"{label:52} accepted")
    except Exception as e:
        print(f"{label:52} refused: {str(e)[:90]}")
print("AnthropicLLMConfig(max_tokens=500) becomes:", p.make_llm_config(AnthropicLLMConfig(max_tokens=500)))

gateway.provider_for('anthropic/claude-haiku-5.5')   refused: 'anthropic/claude-haiku-5.5' is Claude, which the gateway doesn't carry: reach it through 
fallback_models=['claude-haiku-5-5']                 refused: fallback_models can't include Claude (claude-haiku-5-5): Claude never answers in the Chat 
AnthropicLLMConfig(server_tools=[web_search])        refused: server_tools can't be used with a model in the Chat Completions format: they are Anthropic
AnthropicLLMConfig(max_tokens=500) becomes: OpenAIChatLLMConfig(max_tokens=500, effort=None, thinking_tokens=None, temperature=None, response_format=None, fallback_models=None, provider_order=None, caching=None, tool_choice=None, api_kwargs=None)


# 10. Sub-agents pick their own route

When an agent starts a sub-agent, `provider_for_child` asks the router the parent's provider carries. A child on its parent's format keeps the parent's own `for_child` (on Claude Code, a process of its own on the next level's slots); a child on the other format gets that format's provider; a child naming no model stays on its parent's route. Parent and child exchange only the task and the answer, as text, so the two formats never share a history.

In [12]:
from agent_base.common_tools.sub_agent_tool import SubAgentSpec

router = Wire().router()
for parent_model in (CLAUDE, GEMINI):
    parent = router.provider_for(parent_model)
    for child_model in ("claude-haiku-5-5", "openai/gpt-5-nano", None):
        child = provider_for_child(parent, SubAgentSpec(name="c", model=child_model))
        print(f"parent {parent_model:24} child {str(child_model):20} -> {type(child).__name__}")

wire = Wire(claude=[claude_says([{"type": "tool_use", "id": "toolu_1", "name": "spawn_subagent",
                                  "input": {"agent_name": "summarizer", "task": "Summarize: the cat sat."}}], "tool_use"),
                    claude_says([{"type": "text", "text": "The summary: a cat sat."}], "end_turn")],
            chat=[say("A cat sat.")])
parent = AnthropicAgent(system_prompt="Delegate summaries.", model=CLAUDE,
                        provider_value=wire.router().provider_for(CLAUDE),
                        subagents={"summarizer": SubAgentSpec(name="summarizer", description="Summarizes text.",
                                                              system_prompt="Summarize in four words.", model=GEMINI)})
result = await parent.run("Summarize: the cat sat.")
print("\nanswer:", result.final_answer)
print("requests:", [(r.url.host, r.url.path) for r in wire.requests])

parent claude-sonnet-5-5        child claude-haiku-5-5     -> AnthropicProvider


parent claude-sonnet-5-5        child openai/gpt-5-nano    -> GatewayChatProvider
parent claude-sonnet-5-5        child None                 -> AnthropicProvider
parent google/gemini-2.5-flash  child claude-haiku-5-5     -> AnthropicProvider
parent google/gemini-2.5-flash  child openai/gpt-5-nano    -> GatewayChatProvider
parent google/gemini-2.5-flash  child None                 -> GatewayChatProvider


2026-11-18T11:53:49.297098Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


2026-11-18T11:53:49.331118Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T11:53:49.339111Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]



answer: The summary: a cat sat.
requests: [('api.anthropic.com', '/v1/messages'), ('ai-gateway.vercel.sh', '/v1/chat/completions'), ('api.anthropic.com', '/v1/messages')]


# 11. Failures and retries

Rate limits (429), overloads (5xx) and timeouts are retried within the provider's `RetryPolicy`, waiting as long as `retry-after` says (capped at 60 s), and only before anything streamed: after that the step keeps what streamed and fails with `partial_error`, so no text is sent twice. A prompt over the context window is `CONTEXT_OVERFLOW`, which the turn loop answers by compacting; anything else (an unknown model, a model the free tier doesn't cover) is a status error, not retried.

In [13]:
wire = Wire(chat=[chat_error(429, "Too many requests.", "rate_limited"),
                  chat_error(503, "Service temporarily unavailable.", "service_unavailable_error"), say("ok")])
gemini = wire.gateway().provider_for(GEMINI, retry_policy=FAST)
turn = await gemini.generate_stream(system_prompt=None, messages=[Message.user("hi")], tool_schemas=[],
                                    llm_config=None, model=GEMINI, sink=Frames(quiet=True), agent_uuid="nb")
print("after a 429 and a 503:", turn.message.content[0].text, "| requests:", len(wire.requests))

for status, message, kind in ((400, "This model's maximum context length is 1000000 tokens.", "invalid_request_error"),
                              (403, "Free tier users do not have access to this model.", "no_providers_available"),
                              (404, "Model 'nobody/x' not found", "model_not_found")):
    wire = Wire(chat=[chat_error(status, message, kind)])
    provider = wire.gateway().provider_for(GEMINI, retry_policy=FAST)
    try:
        await provider.generate_stream(system_prompt=None, messages=[Message.user("hi")], tool_schemas=[],
                                       llm_config=None, model=GEMINI, sink=Frames(quiet=True), agent_uuid="nb")
    except Exception as e:
        err = provider.classify_error(e)
        print(f"{status}: {err.code.value:18} retriable={err.retriable!s:5} requests={len(wire.requests)}")

2026-11-18T11:53:49.355707Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 429 Too Many Requests" [httpx]


2026-11-18T11:53:49.944457Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 503 Service Unavailable" [httpx]


2026-11-18T11:53:50.337613Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


2026-11-18T11:53:50.342613Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 400 Bad Request" [httpx]


2026-11-18T11:53:50.345613Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 403 Forbidden" [httpx]


2026-11-18T11:53:50.351611Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 404 Not Found" [httpx]


after a 429 and a 503: ok | requests: 3
400: context_overflow   retriable=False requests=1
403: provider_status    retriable=False requests=1
404: provider_status    retriable=False requests=1


# 12. Billing: what Vercel charged

Vercel's price for a model depends on which provider serves the call, and every response reports the charge in its `usage` (`cost`, with `is_byok`). So a gateway step **costs what Vercel charged** (`breakdown["reported_cost"]`), for any model; the price row in `pricing/models.csv` is the fallback, for a step with no reported charge or one on the host's own provider key. Claude is priced from its rows as before.

In [14]:
from agent_base.core.messages import Usage
from agent_base.pricing import calculate_step_cost

million = Usage(input_tokens=1_000_000, output_tokens=1_000_000)
charged = Usage(input_tokens=1_000_000, output_tokens=1_000_000, raw_usage={"cost": 0.0123, "is_byok": False})
own_key = Usage(input_tokens=1_000_000, output_tokens=1_000_000, raw_usage={"cost": 0.0001, "is_byok": True})
for label, usage, model in (("Vercel's charge", charged, GEMINI), ("no charge reported: the row", million, GEMINI),
                            ("the host's own key: the row", own_key, GEMINI), ("a model with no row", charged, "moonshotai/kimi-k3"),
                            ("Claude: its row", million, CLAUDE)):
    cost = calculate_step_cost(usage, model)
    print(f"{label:32} {model:24} ${cost.total_cost:<8} {cost.breakdown if 'reported_cost' in cost.breakdown else ''}")

wire = Wire(chat=[say("Hi.", cost=0.0042)])
agent = AnthropicAgent(system_prompt="Be brief.", model=GEMINI, provider_value=wire.gateway().provider_for(GEMINI))
result = await agent.run("hi")
print("\nthe run's settlement:", result.settlement.turn_cost)

2026-11-18T11:53:50.389261Z [info     ] HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK" [httpx]


Vercel's charge                  google/gemini-2.5-flash  $0.0123   {'reported_cost': 0.0123}
no charge reported: the row      google/gemini-2.5-flash  $2.8      
the host's own key: the row      google/gemini-2.5-flash  $2.8      
a model with no row              moonshotai/kimi-k3       $0.0123   {'reported_cost': 0.0123}
Claude: its row                  claude-sonnet-5-5        $18.0     

the run's settlement: CostBreakdown(total_cost=0.0042, currency='USD', breakdown={'reported_cost': 0.0042}, run_id=None)


# 13. Live: the real gateway (opt-in)

Everything above ran offline. This calls Vercel AI Gateway with `AI_GATEWAY_API_KEY` from `.env`, so it is off by default:

- `RUN_LIVE = True` runs the free models a free-tier account can call (Nemotron 3.5 Lightning Free and Flint v1.0): $0.
- `RUN_CREDIT = True` adds models the free tier's $5 a month covers (Gemini 2.5 Flash, GPT-5 nano): about a cent.

The free models have no zero data retention, so the prompts are made up. For each model: a streamed reply, a tool round replayed in a second run, and a classification, with what Vercel charged.

In [15]:
RUN_LIVE = False
RUN_CREDIT = False

if not RUN_LIVE:
    print("Skipped: set RUN_LIVE = True to call the gateway (free models only, $0).")
else:
    live = VercelGateway.from_env()
    models = ["nvidia/nemotron-3.5-lightning-free", "recursive/flint-v1.0"]
    models += ["google/gemini-2.5-flash", "openai/gpt-5-nano"] if RUN_CREDIT else []
    label_schema = {"type": "json_schema", "json_schema": {"name": "sentiment", "strict": True, "schema": {
        "type": "object", "properties": {"label": {"type": "string", "enum": ["positive", "negative", "neutral"]}},
        "required": ["label"], "additionalProperties": False}}}
    spent = 0.0
    for model in models:
        provider = live.provider_for(model, retry_policy=RetryPolicy(max_retries=3, base_delay=2.0))
        turn = await provider.generate_stream(system_prompt="Answer in one short sentence.",
                                              messages=[Message.user("Name a colour of the sky.")], tool_schemas=[],
                                              llm_config=OpenAIChatLLMConfig(max_tokens=2000), model=model,
                                              sink=Frames(quiet=True), agent_uuid="nb")
        spent += float(turn.message.usage.raw_usage.get("cost") or 0)
        agent = AnthropicAgent(model=model, provider_value=provider, tools=[add],
                               config=OpenAIChatLLMConfig(max_tokens=4000),
                               system_prompt="Use the add tool for every sum. Reply with the number only.")
        first = await agent.run("What is 19 + 23?")
        second = await agent.run("Now add 1 to that result, with the tool.")
        spent += sum(float((m.usage.raw_usage or {}).get("cost") or 0)
                     for m in agent.agent_config.context_messages if getattr(m, "usage", None))
        label = await provider.generate(system_prompt=("Classify the sentiment of the customer's review. Reply with "
                                                       'JSON only: {"label": "positive" | "negative" | "neutral"}.'),
                                        messages=[Message.user("The soup was cold and the waiter ignored us.")],
                                        tool_schemas=[], llm_config=OpenAIChatLLMConfig(max_tokens=2000,
                                                                                         response_format=label_schema),
                                        model=model)
        spent += float(label.message.usage.raw_usage.get("cost") or 0)
        print(f"{model:36} reply {turn.message.content[-1].text[:30]!r:34} tools {first.final_answer!r} -> "
              f"{second.final_answer!r}  label {label.message.content[-1].text}")
    print(f"\nVercel charged ${spent:.6f} in all")
    await live.aclose()

Skipped: set RUN_LIVE = True to call the gateway (free models only, $0).
